# grapheme-aware String Distance 

A **grapheme-aware** string distance library that wraps [`textdistance`](https://github.com/life4/textdistance) with proper Unicode grapheme cluster handling via `graphemes_plusplus.Graphemizer`.

## Setup


In [1]:
# Install dependencies
# pip install textdistance graphemes-plusplus

from pathlib import Path
import sys

project_root = Path.cwd()
for candidate in (project_root, project_root.parent, project_root.parent.parent):
    if (candidate / 'src').exists():
        project_root = candidate
        break

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from graphemes_plusplus.distance import damerau_levenshtein, hamming, jaro, jaro_winkler, levenshtein, longest_common_subsequence


['ශ්\u200d', 'රී', ' ', 'ලං', 'කා', 'වේ', ' ', 'ප්\u200d', 'ර', 'ධා', 'න', ' ', 'ජා', 'ති', 'ය', ' ', 'ව', 'න']


---
## Levenshtein Distance

**Operations:** insert, delete, substitute  
**Returns:** minimum number of single grapheme++ edits to transform `s1` into `s2`


In [9]:

# Example 1: Grapheme deletion (Sinhala)
print("Deletion")
print(f'levenshtein("ක්‍රම", "කම") = {levenshtein("ක්‍රම", "කම")}')
print()

# Example 2: Grapheme insertion (Sinhala)
print("Insertion")
print(f'levenshtein("කම", "ක්‍රම") = {levenshtein("කම", "ක්‍රම")}')
print()

# Example 3: Grapheme substitution (Sinhala)
print("Substitution")
print(f'levenshtein("කම", "ගම") = {levenshtein("කම", "ගම")}')
print()


# Example 4: Identical strings
print("Identical strings")
print(f'levenshtein("මම", "මම") = {levenshtein("මම", "මම")}')

Deletion
levenshtein("ක්‍රම", "කම") = 1

Insertion
levenshtein("කම", "ක්‍රම") = 1

Substitution
levenshtein("කම", "ගම") = 1

Identical strings
levenshtein("මම", "මම") = 0


---
## Hamming Distance

**Compares:** grapheme at position 1 with grapheme at position 1, position 2 with position 2, and so on.  
**Returns:** number of positions that differ; extra trailing graphemes are counted as differences.  
**Best for:** strings where position matters, such as fixed-width labels, codes, and aligned OCR output.

Hamming distance is useful when you do not want edits to shift the rest of the string. A substitution changes one position, but an insertion or deletion can make many later positions differ.


In [3]:
print("Identical Tamil grapheme")
print(f'hamming("ஸ்ரீ", "ஸ்ரீ") = {hamming("ஸ்ரீ", "ஸ்ரீ")}')
print()

print("Classic aligned examples")
print(f'hamming("karolin", "kathrin") = {hamming("karolin", "kathrin")}')
print(f'hamming("1011101", "1001001") = {hamming("1011101", "1001001")}')
print()

print("Unequal grapheme counts")
print(f'hamming("රැ", "රැහ") = {hamming("රැ", "රැහ")}')
print(f'hamming("එවන්න", "") = {hamming("එවන්න", "")}')


Identical Tamil grapheme
hamming("ஸ்ரீ", "ஸ்ரீ") = 0

Classic aligned examples
hamming("karolin", "kathrin") = 3
hamming("1011101", "1001001") = 2

Unequal grapheme counts
hamming("රැ", "රැහ") = 1
hamming("එවන්න", "") = 4


---
## Damerau-Levenshtein Distance

**Operations:** insert, delete, substitute, transpose two adjacent graphemes.  
**Returns:** minimum number of grapheme++ edits, with adjacent swaps counted as one edit.  
**Best for:** typo-tolerant matching where swapped neighboring graphemes are common.

This is close to Levenshtein, but it gives a cheaper score to transposition errors such as `teh` instead of `the`.


In [4]:
pairs = [
    ("ab", "ba"),
    ("the", "teh"),
    ("කම", "මක"),
    ("kitten", "sitting"),
]

header = f"{'s1':<10} {'s2':<10} {'Lev':>4} {'D-L':>4}"
print(header)
print("-" * len(header))
for s1, s2 in pairs:
    print(f"{s1:<10} {s2:<10} {levenshtein(s1, s2):>4} {damerau_levenshtein(s1, s2):>4}")


s1         s2          Lev  D-L
-------------------------------
ab         ba            2    1
the        teh           2    1
කම         මක            2    1
kitten     sitting       3    3


---
## Jaro Similarity

**Range:** `0.0` means no similarity; `1.0` means identical.  
**Compares:** matching graphemes within a moving window, then penalizes transpositions.  
**Best for:** short strings such as names, identifiers, and record-linkage fields.

Jaro is a similarity score, not a distance. Higher is better.


In [5]:
pairs = [
    ("MARTHA", "MARHTA"),
    ("DIXON",  "DICKSONX"),
    ("කරණ", "කරම"),
    ("JELLYFISH", "SMELLYFISH"),
    ("abc", "abc"),
    ("abc", "xyz"),
]
print(f"{'s1':<14} {'s2':<14} {'Jaro':>6}")
print("-" * 36)
for s1, s2 in pairs:
    print(f"{s1:<14} {s2:<14} {jaro(s1, s2):>6.4f}")


s1             s2               Jaro
------------------------------------
MARTHA         MARHTA         0.9444
DIXON          DICKSONX       0.7667
කරණ            කරම            0.7778
JELLYFISH      SMELLYFISH     0.8963
abc            abc            1.0000
abc            xyz            0.0000


---
## Jaro-Winkler Similarity

**Range:** `0.0` to `1.0`, like Jaro.  
**Adds:** a prefix boost for strings that begin with the same graphemes.  
**Best for:** names, titles, and short fields where early graphemes are especially meaningful.

Use Jaro-Winkler when a shared beginning should make two strings feel closer than plain Jaro would score them.


In [6]:
pairs = [
    ("MARTHA", "MARHTA"),
    ("DIXON",  "DICKSONX"),
    ("කරණ", "කරම"),
    ("prefix-match", "prefix-other"),
]
print(f"{'s1':<18} {'s2':<18} {'Jaro':>6} {'Jaro-W':>7}")
print("-" * 52)
for s1, s2 in pairs:
    print(f"{s1:<18} {s2:<18} {jaro(s1,s2):>6.4f} {jaro_winkler(s1,s2):>7.4f}")

print()
print("Jaro-Winkler is usually >= Jaro when the strings share a prefix.")


s1                 s2                   Jaro  Jaro-W
----------------------------------------------------
MARTHA             MARHTA             0.9444  0.9611
DIXON              DICKSONX           0.7667  0.8133
කරණ                කරම                0.7778  0.8222
prefix-match       prefix-other       0.8333  0.9000

Jaro-Winkler is usually >= Jaro when the strings share a prefix.


---
## Longest Common Subsequence (LCS)

**Returns:** length of the longest grapheme sequence that appears in both strings in the same order.  
**Allows:** gaps; the shared graphemes do not need to be next to each other.  
**Best for:** diff-style comparisons, partial overlap, and measuring preserved order.

LCS is about what the two strings keep in common, rather than the number of edits needed to transform one into the other.


In [7]:
print("ABCBDAB vs BDCABA:", longest_common_subsequence("ABCBDAB", "BDCABA"))  # 4
print("hello   vs hello: ", longest_common_subsequence("hello",   "hello"))   # 5
print("abc     vs xyz:   ", longest_common_subsequence("abc",     "xyz"))     # 0
print("කරණ    vs කරම:  ", longest_common_subsequence("කරණ", "කරම"))        # 2
print("ක්‍රම  vs කම:   ", longest_common_subsequence("ක්‍රම", "කම"))        # 1


ABCBDAB vs BDCABA: 4
hello   vs hello:  5
abc     vs xyz:    0
කරණ    vs කරම:   2
ක්‍රම  vs කම:    1


---
## Side-by-Side Comparison

The distance metrics get larger as strings become more different. The similarity metrics get closer to `1.0` as strings become more alike.


In [8]:
test_pairs = [
    ("kitten",  "sitting"),
    ("the",     "teh"),
    ("MARTHA",  "MARHTA"),
    ("ஸ்ரீ",    "ஸ்ரி"),
    ("abc",     "abc"),
    ("abc",     "xyz"),
]

header = f"{'s1':<12} {'s2':<12} {'Lev':>4} {'DL':>4} {'Jaro':>6} {'J-W':>6} {'LCS':>4}"
print(header)
print("-" * len(header))
for s1, s2 in test_pairs:
    lev = levenshtein(s1, s2)
    dl  = damerau_levenshtein(s1, s2)
    j   = jaro(s1, s2)
    jw  = jaro_winkler(s1, s2)
    lcs = longest_common_subsequence(s1, s2)
    print(f"{s1:<12} {s2:<12} {lev:>4} {dl:>4} {j:>6.3f} {jw:>6.3f} {lcs:>4}")


s1           s2            Lev   DL   Jaro    J-W  LCS
------------------------------------------------------
kitten       sitting         3    3  0.746  0.746    4
the          teh             2    1  0.556  0.556    2
MARTHA       MARHTA          2    1  0.944  0.961    5
ஸ்ரீ         ஸ்ரி            2    2  0.000  0.000    0
abc          abc             0    0  1.000  1.000    3
abc          xyz             3    3  0.000  0.000    0


---
## Choosing the Right Metric

| Metric | Output | Best for |
|--------|--------|----------|
| **Levenshtein** | integer distance | General edit distance: insertions, deletions, substitutions |
| **Hamming** | integer distance | Position-by-position differences in aligned strings |
| **Damerau-Levenshtein** | integer distance | Edit distance when adjacent swaps should be cheap |
| **Jaro** | float similarity | Short strings where matching and transposition both matter |
| **Jaro-Winkler** | float similarity | Short strings where a shared prefix should help the score |
| **LCS** | integer length | Ordered overlap, diff-like comparisons, preserved subsequences |

**Key principle:** every function compares *grapheme++ clusters*, not raw Unicode code points. That is what makes the distances meaningful for Sinhala, Tamil, Arabic, emoji, and other multi-codepoint writing systems.
